# CPU Hidden-State Behavioral Walkthrough

This notebook tries the real MIA/vLLM path on a local CPU backend.

Run it only after building vLLM CPU locally and installing this repo with `pip install -e . --no-deps`.

## 1. Import Check

If this cell fails, the problem is at the `import` stage.

In [ ]:
import multiprocessing as mp
import os
from pathlib import Path

import torch
import vllm
from vllm import SamplingParams

from mia import MiaLLM, register_plugins

print('vllm', getattr(vllm, '__version__', 'unknown'))
print('torch', torch.__version__)

## 2. Shared Settings

Use a tiny model and a short context so the fixture stays about behavior, not performance.

In [ ]:
mp.set_start_method('spawn', force=True)
os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD', 'spawn')

REPO_ROOT = Path.cwd()
MODEL_ID = os.environ.get('MIA_CPU_MODEL', 'facebook/opt-125m')
CONFIG_FILE = REPO_ROOT / 'model_configs' / 'hidden_states' / 'Qwen2-1.5B-Instruct.json'
HOOK_DIR = REPO_ROOT / 'tests' / 'cpu_behavioral' / '.hooks'

MODEL_ID, CONFIG_FILE

## 3. Start the CPU Engine

If this cell fails, classify it as `engine-start` or `hook-registration`, depending on the error.

In [ ]:
register_plugins()

llm = MiaLLM(
    model=MODEL_ID,
    worker_name='capture_hs',
    analyzer_name='hidden_states',
    config_file=str(CONFIG_FILE),
    hook_dir=str(HOOK_DIR),
    dtype=torch.float32,
    max_model_len=128,
    gpu_memory_utilization=0.25,
    enable_hook=True,
    enable_prefix_caching=False,
    enforce_eager=True,
)

print('engine started')

## 4. Generate and Capture

If generation works but probes are missing, classify it as `capture`.

In [ ]:
prompts = [
    'Hidden states test prompt one.',
    'Hidden states test prompt two.',
]

outputs = llm.generate(
    prompts,
    SamplingParams(temperature=0.0, max_tokens=1),
    use_hook=True,
)

assert len(outputs) == len(prompts)
assert getattr(outputs[0], 'probes', None) is not None
print('captured probes')

## 5. Analyze Hidden States

If this cell fails after probes exist, classify it as `analyzer`.

In [ ]:
stats = llm.analyze(analyzer_spec={'reduce': 'none'}, probes=outputs[0].probes)
hidden_states = stats['hidden_states']
hidden_size = llm.llm.llm_engine.model_config.hf_config.hidden_size

for layer_name, tensors in sorted(hidden_states.items()):
    assert len(tensors) == len(prompts)
    for tensor in tensors:
        assert tensor.device.type == 'cpu'
        assert tensor.shape == torch.Size([hidden_size])
        assert torch.isfinite(tensor).all()
    print(layer_name, len(tensors), tuple(tensors[0].shape))

## 6. Shutdown

In [ ]:
llm.llm_engine.engine_core.shutdown()
print('done')